In [32]:
#| default_exp pca_module

# PCA on data divided by stature

## Import libraries

In [33]:
#| export 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA

## Plots

### Scree-plot:

Shows the procentages of variance explained by the different PCs

In [34]:
#| export 
def scree_plot(data, subset: list, subset_name: str):
    df_subset = data[subset]
    pca = PCA(n_components=len(df_subset.columns))
    principal_components = pca.fit_transform(df_subset)

    per_var = np.round(pca.explained_variance_ratio_ * 100, decimals=1)
    labels = ['PC' + str(x) for x in range(1, len(per_var) + 1)]

    # Print explained variance
    df_per_var = pd.DataFrame(data=per_var).T
    df_per_var.columns = labels
    print(df_per_var)

    # Find number of PCs that explain at least 95%
    total = 0
    for i in range(len(per_var)):
        total += per_var[i]
        if total >= 95:
            print('Interesting PCs to get 95%: ' + str(labels[:i + 1]))
            break


    # Scree plot with line following the bars
    bars_to_plot = min(10, len(per_var))
    x_vals = list(range(1, bars_to_plot + 1))
    y_vals = per_var[:bars_to_plot]

    plt.figure(figsize=(8, 6))
    plt.bar(x_vals, y_vals, tick_label=labels[:bars_to_plot], alpha=0.8, color='skyblue')
    plt.plot(x_vals, y_vals,  linestyle='--',color='darkblue', linewidth=2)  # This is the line that "follows" the bars

    plt.ylabel('Percentage of Explained Variance',fontsize=16)
    plt.xlabel('Principal Component',fontsize=16)
    plt.title('Scree Plot: ' + subset_name,fontsize=18)
    plt.xticks(rotation=90)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


### Biplot:

Shows the 2D-plot with arrows representing the variables

In [35]:
#| export 
def PC2_plot_biplot(data,data_scaled,subset: list,subsetname: str,compare: str, ax = None ): #
   
    ansur = data
    ansur_c=ansur.copy()
    df_subset = data_scaled[subset]
 
    # PCA with n components
    pca = PCA(n_components=len(df_subset.keys()))
    X_pca = pca.fit_transform(df_subset) # fit: Calculates PCA based on df_subset, transform --> making the new PC-axis 
    
    # Making a DataFrame of the PCA
    df_pca = pd.DataFrame(X_pca, columns= [f'PC{i+1}' for i in range(len(df_subset.keys()))])
    ansur_c = ansur_c.reset_index(drop=True) 
    df_pca[compare]=ansur_c[compare]

    
    df_loadings = pd.DataFrame(pca.components_, columns=df_subset.keys(), index=[f'PC{i+1}' for i in range(len(df_subset.keys()))])
    per_var = np.round(pca.explained_variance_ratio_* 100, decimals=1)

    df_loadings_val=df_loadings.T

    #Gives the top 10 loadigns with largest values for PC1
    top_10 = df_loadings_val['PC1'].abs().nlargest(10)
  

    plt.figure(figsize=(12,10))
    sns.scatterplot(x=df_pca["PC1"], y=df_pca["PC2"], hue=df_pca[compare], alpha=0.6, palette="coolwarm")
    plt.xlabel('PC1 - {0}%'.format(per_var[0]))
    plt.ylabel('PC2 - {0}%'.format(per_var[1]))
    plt.title("PCA: Difference between "+ compare +' - ' +subsetname,fontsize=16)
    plt.axhline(0, color='gray', linestyle='--')
    plt.axvline(0, color='gray', linestyle='--')
    
    if subsetname == 'All':
        scale = 0.006
    elif subsetname == 'Trunk':
        scale = 0.005
    else:
        scale = 0.004

        
    for name in top_10.index:
        x = df_loadings_val.loc[name, 'PC1'] * scale #0.006
        y = df_loadings_val.loc[name, 'PC2'] * scale #0.006
        plt.arrow(0, 0, x, y, color='red', alpha=0.5, width=0.00003)
        plt.text(x + 0.0002, y, name, color='red', fontsize=13)
            
    
    plt.legend(title= compare,loc="upper left", bbox_to_anchor=(1, 1))
    plt.grid()
    plt.show()




### Loadings:

Function thet gives the loading for the PCs from the PCA

In [36]:
#| export 
def loadings_function(data, subset: list):
    ansur = data.copy()
    df_subset = ansur[subset]
    
    #Gives the PCA for all the measurments in the subset for the data
    pca_subset = PCA(n_components=len(df_subset.keys()))
    principal_components = pca_subset.fit_transform(df_subset)

    # Create a DataFrame with the information of the loadings
    df_loadings_subset = pd.DataFrame(pca_subset.components_, columns=df_subset.keys(), index=[f'PC{i+1}' for i in range(len(df_subset.keys()))])
    return df_loadings_subset

Shows a barplot of the top 10 most influencing loadings and their values

Here "number" is number of the PC that will be plotted

In [37]:
#| export 
def plot_loadings(data,subset: list,subsetname:str, number:int):
    df_loadings = loadings_function(data,subset)
    df_loadings_val=df_loadings.T

    top_10_values = df_loadings_val.nlargest(10, f'PC{number}')
   
    sns.barplot(x=top_10_values.T.keys(), y=top_10_values[f'PC{number}'])
    plt.title(f'Value of top {len(top_10_values.T.keys())} loadings with biggest influence - {subsetname}')
    plt.ylabel(f'PC{number}')
    plt.xlabel('Measurement')
    plt.axhline(0, color='black', linewidth=0.8)  # Lägg till en linje vid y=0
    plt.xticks(rotation=70)
    plt.show()



In [38]:
#| export 
def plot_loadings2(data,subset: list,subsetname:str, number:int):
    df_loadings = loadings_function(data,subset)
    df_loadings_val=df_loadings.T

    top_10_values = df_loadings_val.nlargest(10, f'PC{number}')
   
    sns.barplot(x=top_10_values[f'PC{number}'],y=top_10_values.T.keys())
    plt.title(f'Value of top {len(top_10_values.T.keys())} loadings with biggest influence - {subsetname}')
    plt.xlabel(f'PC{number}')
    plt.ylabel('Measurement')
    #plt.axhline(0, color='black', linewidth=0.8)  # Lägg till en linje vid y=0
    #plt.xticks(rotation=70)
    plt.show()


In [39]:
#| export 
def loadings_table(data, subset: list, subsetname: str, number: int):
    df_loadings = loadings_function(data, subset)
    df_loadings_val = df_loadings.T

    top_values = df_loadings_val.nlargest(10, f'PC{number}')
    top_values = top_values[[f'PC{number}']]

    # Skapa innehållet för tabellen som lista av [namn, värde]
    cell_text = [[index, f"{value:.6f}"] for index, value in zip(top_values.index, top_values[f'PC{number}'])]

    # Skapa figuren
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.axis('off')

    table = ax.table(cellText=cell_text,
                     colLabels=['Measurement', f'PC{number}'],
                     cellLoc='center',
                     loc='center')

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)

    plt.title(f'Top {len(top_values)} loadings for PC{number} - {subsetname}',
              fontsize=12, fontweight='bold', pad=20, loc='center')
    plt.tight_layout()
    plt.show()

In [40]:
import nbdev; nbdev.nbdev_export()